# Pytorch의 nn.Embedding
- Pytorch의 Embedding Layer는 word2vec과 마찬가지로 word embedding vector를 찾는 **Lookup Table**이다.
    - 단어의 **정수의 고유 index**가 입력으로 들어오면 Embedding Layer의 **그 index의 Vector**를 출력한다.
    - 모델이 학습되는 동안 모델이 풀려는 문제에 맞는 값으로 Embedding Layer의 vector들이 업데이트 된다.
    - Word2Vec의 embedding vector학습(정적)을 nn.Embedding은 자신이 포함된 모델을 학습 하는 과정에서 한다고 생각하면 된다.

In [43]:
import torch
import torch.nn as nn

embedding_layer = nn.Embedding(
    num_embeddings=20000, # vocab size(총 어휘 수). 20_000개 단어(토큰)에 대한 embedding vector를 만들겠다.(학습 전 미리 생성하는 것)
    embedding_dim=100, # embedding vector의 차원 수. 하나의 토큰을 몇 개의 숫자로 분산 표현할 것인지.
    padding_idx=0 # Padding 토큰의(<PAD>) 어휘사전에서의 index -> 학습하지 않음.
)
# 20_000 x 100 짜리 Embedding Vector 생성

In [44]:
w = embedding_layer.weight
print(w.shape)

torch.Size([20000, 100])


In [45]:
x = [20, 70, 100, 0, 0]
ev = embedding_layer.weight[[20, 70, 100, 0, 0]]
ev.shape

torch.Size([5, 100])

In [3]:
# 3번 토큰 단어의 embedding vector 조회
w[2]

tensor([-0.9050, -0.5400, -1.3085,  0.8615, -1.2368, -1.7254, -1.8288,  0.7219,
         0.0920,  1.1965,  1.2431,  2.3542, -0.9771, -2.3284,  0.6648, -0.3593,
        -1.1924, -0.1912, -0.6930,  0.4046, -0.7510,  0.2805, -1.1479,  2.2397,
        -0.0986, -0.6918,  0.2009,  1.4960,  1.1469, -0.1118,  1.9024,  1.2978,
        -0.8346, -0.3464, -0.6839, -0.2547,  0.4907, -1.1226,  1.2294, -0.5243,
        -0.8825, -1.6729,  2.1031,  0.4177,  0.3209,  1.6892, -0.2371, -0.6017,
         0.0497,  0.8303, -0.8156,  0.4177,  0.7976, -1.1467,  0.4830, -0.8613,
         1.7026,  0.2582,  0.4137, -0.1012,  0.6204, -0.0710, -0.4321, -0.3631,
        -0.3611, -0.4342, -1.5851, -1.2072,  0.5272,  0.9020, -1.2686,  0.5221,
        -1.2196,  1.0174,  0.1876, -0.7693,  1.3518, -2.0792,  0.4621,  0.1247,
        -0.3413,  0.9214, -0.5001,  0.7922, -0.6244, -1.1128, -2.0883, -1.5491,
        -0.5653, -0.4388,  1.1687, -0.0087,  0.5131,  0.3587,  0.7350, -0.8184,
         0.3250, -0.0668, -0.5702, -0.56

In [4]:
# 0번 토큰 - <pad>
w[0]

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.], grad_fn=<SelectBackward0>)

# 네이버 영화 댓글 감성분석(Sentiment Analysis)

## 감성분석(Sentiment Analysis) 이란
입력된 텍스트가 **긍적적인 글**인지 **부정적인**인지 또는 **중립적인** 글인지 분석하는 것을 감성(감정) 분석이라고 한다.   
이를 통해 기업이 고객이 자신들의 기업 또는 제품에 대해 어떤 의견을 가지고 있는지 분석한다.

# Dataset, DataLoader 생성

## Korpora에서 Naver 영화 댓글 dataset 가져오기
- Korpora: https://github.com/ko-nlp/Korpora
- NSMC: http://github.com/e9t/nsmc/
    - input: 영화댓글
    - output: 0(부정적댓글), 1(긍정적댓글)
### API
- **corpus 가져오기**
    - `Korpora.load('nsmc')`
- **text/label 조회**
    - `corpus.get_all_texts()` : 전체 corpus의 text들을 tuple로 반환
    - `corpus.get_all_labels()`: 전체 corpus의 label들을 list로 반환
- **train/test set 나눠서 조회**
    - `corpus.train`
    - `corpus.test`
    - `LabeledSentenceKorpusData` 객체에 text와 label들을 담아서 제공.
        - `LabeledSentenceKorpusData.texts`: text들 tuple로 반환.
        - `LabeledSentenceKorpusData.labels`: label들 list로 반환.

## 데이터 로딩

In [1]:
from Korpora import Korpora
corpus = Korpora.load("nsmc")


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at C:\Users\Playdata\Korpora\nsmc\ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at C:\Users\

In [2]:
all_text = corpus.get_all_texts()
all_label = corpus.get_all_labels()

all_text[:5]

('아 더빙.. 진짜 짜증나네요 목소리',
 '흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나',
 '너무재밓었다그래서보는것을추천한다',
 '교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정',
 '사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 던스트가 너무나도 이뻐보였다')

In [7]:
all_label[:5] # 0: 부정, 1: 긍정

[0, 1, 0, 0, 1]

In [17]:
len(all_text)

200000

In [3]:
# train/test set 조회
corpus.train

NSMC.train: size=150000
  - NSMC.train.texts : list[str]
  - NSMC.train.labels : list[int]

In [18]:
corpus.test
# corpus.test.texts[:5]
# corpus.test.labels[:5]

NSMC.test: size=50000
  - NSMC.test.texts : list[str]
  - NSMC.test.labels : list[int]

## 토큰화
1. 형태소 단위 token화
    - konlpy로 token화 한 뒤 다시 한 문장으로 만든다.
2. 1에서 처리한 corpus를 BPE 로 token화
   
### 전처리 함수

#### 형태소 단위 분절

In [4]:
from kiwipiepy import Kiwi
import string
import re

kiwi = Kiwi()
def text_preprocessing(text):
    """
    1. 영문 -> 소문자로 변환
    2. 구두점 제거
    3. 형태소 기반 토큰화
    4. 형태소로 토큰화 한 뒤 다시 하나의 문자열로 묶어서 반환.
    """
    # 1. 
    text = text.lower()
    # 2. 특수문자 -> 공백
    text = re.sub(rf"[{string.punctuation}]", " ", text)
    # 3. 
    text = [token.lemma for token in kiwi.tokenize(text)]
    4. 
    return " ". join(text)

In [5]:
print(all_text[20])
text_preprocessing(all_text[20])

나름 심오한 뜻도 있는 듯. 그냥 학생이 선생과 놀아나는 영화는 절대 아님


'나름 심오 하 ᆫ 뜻 도 있다 는 듯 그냥 학생 이 선생 과 놀아나다 는 영화 는 절대 아니다 ᆷ'

In [13]:
import string
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [22]:
### 전처리
train_inputs = [text_preprocessing(txt) for txt in corpus.train.texts] # trainset 댓글 - 전처리
test_inputs = [text_preprocessing(txt) for txt in corpus.test.texts] # testset 댓글 - 전처리

train_labels = corpus.train.labels
test_labels = corpus.test.labels

In [23]:
# 피클로 저장
import os 
os.makedirs("data/nsmc", exist_ok=True)

train_data = {"text": train_inputs, "label": train_labels}
test_data = {"text": test_inputs, "label": test_labels}

import pickle
with open("data/nsmc/preprocessing_train.pkl", "wb") as fo:
    pickle.dump(train_data, fo)

with open("data/nsmc/preprocessing_test.pkl", "wb") as fo:
    pickle.dump(test_data, fo)

In [24]:
import pickle
with open("data/nsmc/preprocessing_train.pkl", "rb") as fi:
    train_data = pickle.load(fi)

with open("data/nsmc/preprocessing_test.pkl", "rb") as fi:
    test_data = pickle.load(fi)

train_inputs, train_labels = train_data["text"], train_data["label"]
test_inputs, test_labels = test_data["text"], test_data["label"]

# 실제론 모듈화 해서 호출이 good

In [25]:
len(train_inputs), len(test_inputs)

(150000, 50000)

In [26]:
# 토큰화를 위해 text 병합
all_inputs = train_inputs + test_inputs

### 토큰화
- Subword 방식 토큰화 적용
- Byte Pair Encoding 방식으로 huggingface tokenizer 사용
    - BPE: 토큰을 글자 단위로 나눈뒤 가장 자주 등장하는 글자 쌍(byte paire)를 찾아 합친뒤 어휘사전에 추가한다.
    - https://huggingface.co/docs/tokenizers/quicktour
    - `pip install tokenizers`

In [27]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

vocab_size = 30_000

tokenizer = Tokenizer(
    BPE(unk_token="<unk>")
)
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(
    vocab_size=vocab_size,
    min_frequency=5,
    special_tokens=["<unk>", "<pad>"], # id: 0-<unk>, 1-<pad>
    continuing_subword_prefix="##"
)

# 학습 데이터가 list[doc]
tokenizer.train_from_iterator(all_inputs, trainer=trainer)
# 학습 데이터가 파일 : tokenizer.train([파일경로], ...)


In [28]:
# 저장
import os
os.makedirs("saved_model/nsmc/lstm", exist_ok=True)
tokenizer.save("saved_model/nsmc/lstm/bpe_tokenizer.json")
# Tokenizer.from_file(경로) -> 로드

In [29]:
# 총 어휘 수
tokenizer.get_vocab_size()

24860

In [30]:
idx = 120
print(all_inputs[idx])
encoding = tokenizer.encode(all_text[idx])
# encoding
print("토큰 ID")
print(encoding.ids)
print("토큰 문자열")
print(encoding.tokens)

중국인 특유 의 과장 허풍 있다 어 보이다 려고 안간힘 쓰다 ᆫ 노력 은 가상 하 나 고증 과 현실감 떨어지다 는 설정 이 거북 스럽 다 도대체 그 들 은 왜 이렇다 게 까지 스스로 를 과대 포장 하 는 것 이다 ᆫ지
토큰 ID
[9204, 6157, 3059, 6684, 0, 17565, 0, 0, 0, 23577, 7732, 5766, 1937, 3340, 4341, 1916, 6094, 3016, 9046, 6241, 0, 8023, 3442, 7397, 7545, 3156, 3410, 5653, 3057, 7238, 6905, 0, 0, 0, 5643, 670, 3327, 3016, 2065, 6340, 19436, 6771, 3421, 7760, 3293, 3271, 3068, 3410, 574, 3043, 3156, 0, 0, 0]
토큰 문자열
['중국인', '특유', '##의', '과장', '<unk>', '허풍', '<unk>', '<unk>', '<unk>', '있어', '##보이', '##려고', '안', '##간', '##힘', '쓴', '노력', '##은', '가상', '##하나', '<unk>', '고증', '##과', '현실감', '떨어', '##지', '##는', '설정', '##이', '거북', '##스럽다', '<unk>', '<unk>', '<unk>', '도대체', '그', '##들', '##은', '왜', '이렇게', '##까지', '스스로', '##를', '과대', '##포', '##장', '##하', '##는', '것', '##인', '##지', '<unk>', '<unk>', '<unk>']


## Dataset, DataLoader 생성

In [31]:
import torch
from torch.utils.data import Dataset, DataLoader

class NSMCDataset(Dataset):
    def __init__(self, texts, labels, max_length, tokenizer):
        """
        texts: list - 댓글 리스트. 리스트에 댓글들을 담아서 받는다. ["댓글", "댓글", ...]
        labels: list - Label 리스트. (댓글의 긍부정 여부 - 긍정: 1, 부정: 0)
        max_length: int: 개별 댓글의 최대 token 개수. 모든 댓글의 토큰수를 max_length에 맞춘다.
        tokenizer: tokenizers.Tokenizer
        """
        self.max_length = max_length
        self.tokenizer = tokenizer
        self.labels = labels
        # text는 토큰화해서 저장
        self.texts = [
            self.__pad_token_sequences(self.tokenizer.encode(txt).ids) for txt in texts
        ]

    ###########################################################################################
    # id로 구성된 개별 문장 token list를 받아서 패딩 추가 [20, 2, 1] => [20, 2, 1, 0, 0, 0, ..]
    ############################################################################################
    def __pad_token_sequences(self, token_sequences):
        """
        token id로 구성된 개별 문서(댓글)의 token_id list를 받아서 'max_length 길이에 맞추는' 메소드
        max_length 보다 '토큰수가 적으면 Padding 토큰 추가', '많으면 max_length 크기로 줄인다'.
            ex) max_length=5 이고 pad토큰 id가 0이라면
                [20, 2, 1] => [20, 2, 1, 0, 0]
                [20, 21, 30, 34, 60, 17, 21, 33] -> [20, 21, 30, 34, 60]
        """
        # 1. <pad> 토근 id 조회
        pad_token_id = self.tokenizer.token_to_id("<pad>")
        
        # 2. 입력 문서의 토큰 수
        seq_len = len(token_sequences)

        # 3. max_length 길이에 토큰 수 맞추기
        if self.max_length < seq_len: # 많은 경우
            result = token_sequences[:self.max_length]
        else: # 모자란 경우
            result = token_sequences + ([pad_token_id] * (self.max_length - seq_len))

        return result
        
    def __len__(self):
        """총 데이터 수 반환. len(dataset) 호출 시 반환값"""
        return len(self.labels)

    def __getitem__(self, idx):
        """
        idx 번째 'text와 label'(X, y)을 학습 가능한 type으로 변환해서 반환
        Parameter
            idx: int 조회할 index
        Return
            tuple: (torch.LongTensor, torch.FloatTensor) - x: 댓글 토큰_id를 LongTensor(int64), y: 정답 Label을 FloatTensor(float32)
            - embedding 입력이 LongTensor
        """
        # self.texts : list[list] -> 단어 list[개별 단어 list]
        # self.labels: list[int]
        X = torch.tensor(self.texts[idx], dtype=torch.int64)
        y = torch.tensor(self.labels[idx], dtype=torch.float32).unsqueeze(dim=-1)
        
        return X, y
    

In [32]:
### DataSet 생성
max_length = 30
trainset = NSMCDataset(train_inputs, train_labels, max_length, tokenizer)
testset = NSMCDataset(test_inputs, test_labels, max_length, tokenizer)

len(trainset), len(testset)

(150000, 50000)

In [33]:
print(trainset.texts[3])

[11674, 5379, 5251, 8990, 5502, 5264, 920, 5258, 946, 5291, 8201, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [34]:
# 테스트
c, l = trainset[3]
c, l

(tensor([11674,  5379,  5251,  8990,  5502,  5264,   920,  5258,   946,  5291,
          8201,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1]),
 tensor([0.]))

In [35]:
tokenizer.decode(list(c))

'교도소 이야기 이다 구먼 솔직히 재미 는 없다 다 평점 조정'

In [36]:
### DataLoader
batch_size = 100
train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(testset, batch_size=batch_size)

len(train_loader), len(test_loader)

(1500, 500)

# 모델링
- Embedding Layer를 이용해 Word Embedding Vector를 추출한다.
- LSTM을 이용해 Feature 추출 -> 문서에 대한 feature 추출
- Linear + Sigmoid로 댓글 긍정일 확률 출력
  
![outline](figures/rnn/RNN_outline.png)

## 모델 정의

In [ ]:
# 순전파: embedding layer -> LSTM Layer(문서-댓글에 대한 feature 추출) -> Linear + Sigmoid(분류기)

## 모델 생성

In [37]:
import torch
import torch.nn as nn

class NSMCClassifier(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers=1, bidirectional=True, dropout=0.2, pad_token_id=1):
        super().__init__()
        # EmbeddingLayer -> LSTM -> Linear -> Sigmoid

        # 토큰들(Tensor[int] - [20, 30, 1289, 30, ...]), shape: [batch, max_length] 
        # -> embedding_layer -> 출력 tensor shape: [batch, seq_len, embedding_dim]
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size, # 어휘사전 크기(단어 수)
            embedding_dim=embedding_dim, # Embedding Vector의 차원 수
            padding_idx=pad_token_id # 1
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim, # 개별 단어의 embedding 벡터 차원
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional,
            dropout=0.0 if num_layers==1 else dropout
        )
        
        # classifier의 입력 -> LSTM의 마지막 timestep의 hidden state
        # out, (hidden, cell) = lstm(X) -> out[-1]를 입력
        self.classifier = nn.Linear(
            in_features=hidden_size*2 if bidirectional else hidden_size,
            out_features=1 # 이진 분류 -> 양성일 확률값
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, X): # 입력 h값 0으로 할 거라 안 받음
        # X: 입력 문서 - 토큰 리스트[batch_size, seq_len(max_length)]
        # X -> Embedding
        # 출력 shape: [batch_size, seq_len, embedding_dim]
        embedding_vector = self.embedding(X)

        # embedding vector의 batch와 seq len 위치 변경
        embedding_vector = embedding_vector.transpose(1, 0) # [seq_len, batch_size, em_dim]

        # embedding_vector -> LSTM
        out, _ = self.lstm(embedding_vector) # _ : 안 쓴다는 표시
        # out shape: [seq_len, batch_size, hidden_size * bidirectional(->단방향 1, 양방향 2)]

        # out[-1](마지막 timestep값) -> Classifier(Linear)
        output = self.classifier(out[-1])
        last_output = self.sigmoid(output)

        return last_output

In [ ]:
# 이중 분류 : 고양이가 맞다/아니다 -> 아니다는 부정이 아닌 그저 고양이가 아닌 것. 강아지일 수도, 자동차일 수도 있고 그런 것.
# 다중 분류 : 각 결과값을 확률값으로 바꿔주는 sigmoid

In [38]:
# 하이퍼 파라미터/설정값 정의
device = "cuda" if torch.cuda.is_available() else "cpu"
vocab_size = tokenizer.get_vocab_size()
embedding_dim = 100
hidden_size = 64
num_layers = 1
bidirectional = True

In [39]:
model = NSMCClassifier(
    vocab_size=vocab_size, 
    embedding_dim=embedding_dim, 
    hidden_size=hidden_size, 
    num_layers=num_layers, 
    bidirectional=bidirectional
).to(device)

In [105]:
!uv pip install torchinfo

Resolved 1 package in 113ms
Installed 1 package in 72ms
 + torchinfo==1.8.0


In [ ]:
from torchinfo import summary
# summary(모델, 입력데이터 shape) -> 입력 데이터 타입: float32
#                             model 입력 데이터 타입: int64(LongTensor) => 직접 int64 타입의 더미 데이터 만들어 넣어줌
dummy_input = torch.randint(1, 10, (batch_size, max_length))
summary(model, input_data=dummy_input, device=device)

# embeddig layer : 토큰을 idx로 조회 시 사용되는 번호가 됨
# 문서 -> token id (토큰화) : 나는 밥을 먹었다 = [300, 7, 600]
# [300, 7, 600] : embedding layer에 들어감 -> 300번 idx의 embedding 벡터값, 7번 idx의 embedding 벡터값, 600번 idx의 embedding 벡터값
# 토큰값이 idx - 정수값. float 들어가면 오류

Layer (type:depth-idx)                   Output Shape              Param #
NSMCClassifier                           [100, 1]                  --
├─Embedding: 1-1                         [100, 30, 100]            2,486,000
├─LSTM: 1-2                              [30, 100, 128]            84,992
├─Linear: 1-3                            [100, 1]                  129
├─Sigmoid: 1-4                           [100, 1]                  --
Total params: 2,571,121
Trainable params: 2,571,121
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 503.59
Input size (MB): 0.02
Forward/backward pass size (MB): 5.47
Params size (MB): 10.28
Estimated Total Size (MB): 15.78

In [41]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
loss_fn = nn.BCELoss()

## 학습

### Train/Test 함수 정의

In [52]:
# 1 epoch 학습 함수
def train(model, dataloader, loss_fn, optimizer, device="cpu"):
    # 1. 모델을 train모드
    model.train()
    # 2. Model을 device이동
    model = model.to(device)

    # 3. 1 에폭 학습
    train_loss = 0.0
    for X, y in dataloader:
        # X, y를 device로 이동
        X, y = X.to(device), y.to(device)
        #추론
        pred = model(X)
        # loss 계산
        loss = loss_fn(pred, y)
        # grad 계산
        loss.backward()
        # 파라미터 업데이터
        optimizer.step()
        # 파라미터 grad 초기화
        optimizer.zero_grad()
        # loss값 누적
        train_loss += loss.item()

    return train_loss / len(dataloader)

In [53]:
# 평가 함수
@torch.no_grad
def eval(model, dataloader, loss_fn, device="cpu"):
    # 모델 평가/검증 시 사용
    # 1. eval 모드로 변경
    model.eval()
    model = model.to(device)

    eval_loss, eval_acc = 0.0, 0.0
    for X, y in dataloader:
        # X, y 이동
        X, y = X.to(device), y.to(device)
        # 추론
        pred_proba = model(X) # 양성일 확률
        pred_class = (pred_proba > 0.5).type(torch.int32) # True: 1, False: 0 => Label
                     # torch.where(pred_proba > 0.5) -> true인 idx 알려줌, torch.where(pred_proba > 0.5, 1, 0) -> (조건, True반환값, False반환값)

        # 평가
        eval_loss = loss_fn(pred_proba, y).item()
        eval_acc += (pred_class == y).sum().item()

    return eval_loss / len(dataloader), eval_acc / len(dataloader.dataset)

### Train

In [54]:
import time
epochs = 3

train_loss_list = []
val_loss_list = []
val_acc_list = []

s = time.time()
for epoch in range(epochs):
    train_loss = train(model, train_loader, loss_fn, optimizer, device)
    val_loss, val_acc = eval(model, test_loader, loss_fn, device)
    
    train_loss_list.append(train_loss)
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)
    print(f"Epoch: {epoch}, {train_loss} || {val_loss} || {val_acc}")

e = time.time()
print("걸린 시간: ", e-s, "초")

Epoch: 0, 0.4257355454762777 || 0.000821675181388855 || 0.8075
Epoch: 1, 0.39250212955474856 || 0.0007430716753005982 || 0.81836
Epoch: 2, 0.37069319348533947 || 0.0007346088290214539 || 0.82602
걸린 시간:  125.74745655059814 초


## 모델저장

In [55]:
torch.save(model, "saved_model/nsmc/lstm/model.pth")

# load_model = torch.load("saved_model/nsmc/lstm/model.pth", weights_only=False)

# 서비스

## 전처리 함수들

In [63]:
from kiwipiepy import Kiwi
import string
import re

kiwi = Kiwi()
def text_preprocessing(text):
    """
    1. 영문 -> 소문자로 변환
    2. 구두점 제거
    3. 형태소 기반 토큰화
    4. 형태소로 토큰화 한 뒤 다시 하나의 문자열로 묶어서 반환.
    """
    text = text.lower()
    text = re.sub(rf"[{string.punctuation}]", ' ', text) # 구두점(특수문자)들을 ' '으로 변환.
    text = [token.lemma for token in kiwi.tokenize(text)] # [a, b, c,]
    return ' '.join(text)

In [64]:
def pad_token_sequences(token_sequences, max_length):
    """padding 처리 메소드."""
    pad_token = tokenizer.token_to_id('<pad>')  
    seq_length = len(token_sequences)
    if seq_length > max_length:                 
        result = token_sequences[:max_length]
    else:                                            
        result = token_sequences + ([pad_token] * (max_length - seq_length))
    return result

In [65]:
def predict_data_preprocessing(
        text_list:list[str], 
        tokenizer, 
        max_length:int=30
        ):
    """
    모델에 입력할 수있는 input data를 생성
    Parameter:
        text_list: list - 추론할 댓글리스트
    Return
        torch.LongTensor - 댓글 token_id tensor
    """
    # 기본적인 text 전처리 - ["댓글1", "댓글2", ...] -> ["전처리된 댓글1", "전처리된 댓글2", ...]
    text_list = [text_preprocessing(txt) for txt in text_list]

    # 토큰화
    # ["전처리된 댓글1", "전처리된 댓글2", ...] -> [[100, 230, 170]]
    token_list = [tokenizer.encode(txt).ids for txt in text_list]

    # max_length 크기에 토큰 개수 맞추기
    # [[100, 230, 170]] -> [[100, 230, 170, 1, 1, 1, 1, 1]]
    token_list = [pad_token_sequences(tokens, max_length) for tokens in token_list]

    return torch.tensor(token_list, dtype=torch.int64)

## 추론

In [66]:
comment_list = ["아 진짜 재미없다.", "여기 식당 먹을만 해요", "이걸 영화라고 만들었냐?", "기대 안하고 봐서 그런지 괜찮은데.", "이걸 영화라고 만들었나?", "아! 뭐야 진짜.", "재미있는데.", "연기 짱 좋아. 한번 더 볼 의향도 있다.", "뭐 그럭저럭"]

In [68]:
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file("saved_model/nsmc/lstm/bpe_tokenizer.json")

input_tensors = predict_data_preprocessing(
    comment_list, tokenizer, 30
)
print(input_tensors.shape)

torch.Size([9, 30])


In [69]:
input_tensors[5]

tensor([1935, 1404, 5251, 1964, 5278,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1])

In [ ]:
proba = torch.tensor([0.1, 0.7, 0.9])
label = torch.where(proba > 0.5, 1, 0)
for l, p in zip(label, proba):
    print(l, p)

# 0일 확률 0.9가 되어야 하는데 확률값 그대로 나옴

tensor(0) tensor(0.1000)
tensor(1) tensor(0.7000)
tensor(1) tensor(0.9000)


In [72]:
proba = torch.tensor([0.1, 0.7, 0.9])
label = torch.where(proba > 0.5, 1, 0)
proba2 = [1-p if l==0 else p for l, p in zip(label, proba)]
for l, p in zip(label, proba2):
    print(l, p)

# 0일 확률 0.9가 되어야 하는데 확률값 그대로 나옴

tensor(0) tensor(0.9000)
tensor(1) tensor(0.7000)
tensor(1) tensor(0.9000)


In [79]:
# 추론 함수
@torch.no_grad
def predict(model, comment_list:torch.LongTensor, device="cpu"):
    model.eval()
    model = model.to(device)
    comment_list = comment_list.to(device)
    
    pred_proba = model(comment_list) # 양성(긍정)일 확률
    pred_label = torch.where(pred_proba > 0.5, 1, 0) # 부정 : 0, 긍정 : 1
    result_proba = [1-p if l==0 else p for l, p in zip(pred_label, pred_proba)]

    return pred_label, result_proba

In [82]:
device = "cuda" if torch.cuda.is_available() else "cpu"
load_model = torch.load("saved_model/nsmc/lstm/model.pth", weights_only=False)
p_label, p_proba = predict(load_model, input_tensors, device)

In [83]:
for txt, label, proba in zip(comment_list, p_label, p_proba):
    print(txt)
    print("긍정적" if label.item() == 1 else "부정적")
    print("확률 : ", proba.item())
    print("#"*50)

아 진짜 재미없다.
부정적
확률 :  0.9858063459396362
##################################################
여기 식당 먹을만 해요
부정적
확률 :  0.620689868927002
##################################################
이걸 영화라고 만들었냐?
부정적
확률 :  0.9850606322288513
##################################################
기대 안하고 봐서 그런지 괜찮은데.
긍정적
확률 :  0.782640278339386
##################################################
이걸 영화라고 만들었나?
부정적
확률 :  0.9593244791030884
##################################################
아! 뭐야 진짜.
부정적
확률 :  0.8657883405685425
##################################################
재미있는데.
긍정적
확률 :  0.7543869018554688
##################################################
연기 짱 좋아. 한번 더 볼 의향도 있다.
긍정적
확률 :  0.9706535339355469
##################################################
뭐 그럭저럭
부정적
확률 :  0.9593309164047241
##################################################
